# Credit Risk Platform — Exploratory Data Analysis

**Dataset:** Home Credit Default Risk (Kaggle)
**Tables used:** `application_train.csv`, `bureau.csv`, `previous_application.csv`, `POS_CASH_balance.csv`

This notebook covers:
1. Dataset summary & shape
2. Feature categorization
3. Data quality / missing values
4. Business insights (5+) with supporting charts


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

sns.set_style("whitegrid")
plt.rcParams['figure.facecolor'] = 'white'
pd.set_option('display.max_columns', 50)

## 1. Load Feature Table

This is the output of `src/data/preprocessor.py` — the raw `application_train.csv` joined with aggregated `bureau`, `previous_application`, and `POS_CASH_balance` signals.

In [ ]:
df = pd.read_parquet('../data/features_train.parquet')
print(f"Shape: {df.shape[0]:,} applicants x {df.shape[1]} features")
df.head()

## 2. Dataset Summary & Feature Categorization

In [ ]:
dtype_counts = df.dtypes.value_counts()
print("Feature types:")
print(dtype_counts)

print("\nFeature groups:")
print("  Demographics/application core: ~40 columns (income, family, housing, employment)")
print("  Building/housing quality:      ~50 columns (mostly the *_AVG/_MODE/_MEDI apartment features)")
print("  Bureau (external credit):       9 engineered columns")
print("  Previous applications:          5 engineered columns")
print("  POS/Cash monthly history:       8 engineered columns")
print("  Engineered ratios:              4 columns (credit/income, annuity/income, etc.)")

## 3. Data Quality — Missing Values

A large block of housing/apartment quality columns (`*_AVG`, `*_MODE`, `*_MEDI`) are 50-70% missing — these describe the applicant's building, and many applicants live in housing types where this isn't recorded (e.g. renters). We keep these but rely on the model's native missing-value handling (LightGBM) rather than aggressive imputation, since missingness itself can be informative (e.g. "housing type unknown" correlates with renting).

In [ ]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
print(f"Columns with >40% missing: {(missing > 40).sum()}")
print(f"Columns with 0% missing:   {(missing == 0).sum()}")

img = plt.imread('eda_charts/07_missing_data.png')
plt.figure(figsize=(10,7))
plt.imshow(img)
plt.axis('off')
plt.show()

## 4. Target Distribution — Class Imbalance

This is the single most important data-quality fact for the ML module: only ~8% of applicants default. A naive model can hit 92% "accuracy" by always predicting "no default" — which is useless for a bank. This drives our choice of evaluation metric (ROC-AUC / PR-AUC, not accuracy) and our imbalance-handling strategy (SMOTE + class weighting) in the ML notebook.

In [ ]:
print(df['TARGET'].value_counts())
print(df['TARGET'].value_counts(normalize=True) * 100)

img = plt.imread('eda_charts/01_target_distribution.png')
plt.figure(figsize=(6,4))
plt.imshow(img)
plt.axis('off')
plt.show()

## 5. Business Insight #1 — Younger applicants default more

Default rate falls steadily and almost linearly with age, from ~11.4% (20-30) down to ~4.9% (60-70). This likely reflects income/career stability building over time. **Business implication:** age-banded risk adjustment or requiring additional income verification for younger applicants.

In [ ]:
img = plt.imread('eda_charts/02_default_by_age.png')
plt.figure(figsize=(7,4))
plt.imshow(img)
plt.axis('off')
plt.show()

## 6. Business Insight #2 — Employment/income type matters a lot

Unemployed and Maternity-leave applicants default at 36-40% — dramatically higher than Working (9.6%), State servants (5.8%) or Pensioners (5.4%). **Business implication:** income-type should be a first-class field in underwriting rules, not just a minor feature.

In [ ]:
img = plt.imread('eda_charts/03_default_by_income_type.png')
plt.figure(figsize=(8,4.5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 7. Business Insight #3 — External bureau overdue history nearly doubles risk

Applicants with at least one overdue loan on record at other credit bureaus default at 15.9% vs 8.0% for those with none — almost exactly double. This confirms that pulling external bureau data (not just this lender's own history) adds real predictive signal, justifying the multi-table join.

In [ ]:
img = plt.imread('eda_charts/04_default_by_bureau_overdue.png')
plt.figure(figsize=(6,4))
plt.imshow(img)
plt.axis('off')
plt.show()

## 8. Business Insight #4 — Prior refusal history predicts future default

Applicants who were refused at least once on a previous application with this lender default at 10.3% vs 7.0% for those never refused. This is a useful, immediately actionable signal for underwriting — even a single past refusal materially changes risk.

In [ ]:
img = plt.imread('eda_charts/05_default_by_prior_refusal.png')
plt.figure(figsize=(6,4))
plt.imshow(img)
plt.axis('off')
plt.show()

## 9. Business Insight #5 — Credit-to-income ratio distribution shifts for defaulters

Defaulting applicants skew toward higher credit-to-income ratios — they're borrowing more relative to what they earn. This is one of our strongest engineered features (`CREDIT_INCOME_RATIO`) and is directly usable as a business rule threshold.

In [ ]:
img = plt.imread('eda_charts/06_credit_income_ratio_dist.png')
plt.figure(figsize=(7,4.5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 10. Summary of Key Findings

| # | Insight | Default Rate Comparison |
|---|---------|--------------------------|
| 1 | Age | 20-30: 11.4% vs 60-70: 4.9% |
| 2 | Employment type | Unemployed: 36.4% vs Pensioner: 5.4% |
| 3 | Bureau overdue history | Has overdue: 15.9% vs None: 8.0% |
| 4 | Prior refusal (this lender) | Refused before: 10.3% vs Never: 7.0% |
| 5 | Credit/income ratio | Defaulters skew visibly higher |
| 6 | POS/cash DPD history | Has DPD: 10.1% vs None: 7.6% |
| 7 | Gender | Male: 10.1% vs Female: 7.0% |

**Data quality takeaway:** ~50 housing-quality columns are 50-70% missing (structural, not random — tied to housing type) — handled via the model's native missing-value support rather than imputation, which would inject noise.

**Modeling implication:** severe class imbalance (~8% positive class) means we optimize and evaluate on ROC-AUC/PR-AUC, apply SMOTE + class weighting, and tune the decision threshold for the Low/Medium/High risk bands rather than using a naive 0.5 cutoff — see `notebooks/model_training.ipynb` / `src/ml/train.py`.